In [ ]:
!pip install langchain langchain_core langchain_community pypdf pymupdf sentence-transformers chromadb

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content = "Hello",
    metadata={"source": "https://www.google.com"}
)

In [ ]:
sample_doc

In [ ]:
type(sample_doc)

In [ ]:
# text data convert to document
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("/content/drive/MyDrive/RAG/Python.txt",encoding ="utf-8")

In [ ]:
document= loader.load()

In [ ]:
document

In [ ]:
# convert pdf data to document

from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader = PyPDFLoader("/content/drive/MyDrive/RAG/pdf/research2.pdf")

pdf_document = pdf_loader.load()

pdf_document

# Ingestion Pipeline

In [ ]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [ ]:
def load_all_pdfs():
  folder_path = "/content/drive/MyDrive/RAG/pdf"
  num_docs = 0
  all_docs =[]

  for filename in os.listdir(folder_path):

    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(folder_path,filename) # full path

      loader = PyPDFLoader(pdf_path)
      pdf_doc = loader.load()
      all_docs.extend(pdf_doc)
      num_docs +=1

  print("total pdfs :" , num_docs)
  print("total_pages:" , len(all_docs))


  return all_docs


In [ ]:
all_pdf_doc = load_all_pdfs()

In [ ]:
type(all_pdf_doc[1])

In [ ]:
print(all_pdf_doc[1])

# **Cunking the data in langchain document**

In [ ]:
!pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_doc(document,chunk_size=500 , chunk_overlap =50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(document)

  return chunked_docs

In [ ]:
chunks = chunk_doc(all_pdf_doc)

In [ ]:
chunks

In [ ]:
len(chunks)

# **Embedding**

In [ ]:
from sentence_transformers import SentenceTransformer

# Lets create a class

In [ ]:
class EmbeddingManager:
  #constructure
  def __init__(self, model_name = "all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("loading model...." , self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


  # fun to generate embeddings
  def generate_embeddings(self,text):
    embeddings = self.model.encode(text,show_progress_bar=True)
    print("embeddings shape:", embeddings.shape)
    return embeddings



In [ ]:
embedding_manager = EmbeddingManager()

# **lets create a Vector Store**
(Vector Store - its a small scale version of vector database )

In [ ]:
import os
os.makedirs('/content/vector_store', exist_ok=True)

In [ ]:
import chromadb
import uuid  # create indexs for documents

In [ ]:
# lets create a class for vector store
# path where vector our vector embeddings
# collections == tables

class VectorStoreManager:
  # constructor
  def __init__(self, persist_directory = "/content/vector_store" , collection_name = "pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

    self._initialize_store() # private function

  #
  def _initialize_store(self):
    os.makedirs(self.persist_directory,exist_ok=True)

    #create client
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    # create the collection
    self.collection  = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata ={"description": " vectore store collection for pdf embeddings in RAG "}

    )

    print("initialized the vector stpre with collection : ", self.collection_name)
    print("docs in collection : " , self.collection.count())


  def add_documents(self, documents, embeddings):
    if len(documents)!= len(embeddings):
      raise ValueError("num of documnets does not match num of embeddings")

    # store -> ids, embedding , document ,metadata

    ids=[]
    all_metadata =[]
    documents_content =[]
    embeddings_list =[]

    for i , (doc,embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata ["doc_index"] = i
      metadata["content_length"] = len(doc.page_content)
      all_metadata. append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

      self.collection.add(
          ids=ids,
          metadatas=all_metadata,
          documents=documents_content,
          embeddings=embeddings_list
          )

    print("total documents added in vector store=", len(documents_content))
    print("docs in collection:", self.collection.count())



In [ ]:
vector_store = VectorStoreManager()

In [ ]:
# import shutil

# folder_path = '/content/vector_store'
# shutil.rmtree(folder_path)

In [ ]:
texts = [doc.page_content for doc in chunks]

embedding =embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

In [ ]:
from IPython.display import Javascript

Javascript('''
IPython.notebook.clear_all_output();
''')

In [ ]:
import nbformat

# Load the notebook
file_path = 'RAG_pipeline.ipynb'
with open(file_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Check if widgets exist and add the missing 'state' key
if 'widgets' in nb.metadata:
    for key in nb.metadata.widgets:
        if 'state' not in nb.metadata.widgets[key]:
            nb.metadata.widgets[key]['state'] = {}

# Save it back
with open(file_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)